In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train_full = train_full[target_col]
y_test = test[target_col]

In [5]:
# ============================================================
# 3. 시간사상 Group K-Fold 설정 및 Fold 배정 저장
# ============================================================

from sklearn.model_selection import GroupKFold

# ------------------------------------------------------------
# 3-1. Group 설정
# 같은 시간사상에 속한 관측치는 반드시 같은 Fold에 들어가도록 설정
# ------------------------------------------------------------

kf_groups = train_full['event_id']

print(f'<시간사상 Group 확인>')
print(f'전체 학습 행 수   : {len(train_full):,}')
print(f'시간사상 개수     : {kf_groups.nunique():,}')
print()

assert kf_groups.notnull().all(), 'event_id 결측치 존재'


# ------------------------------------------------------------
# 3-2. GroupKFold 정의
# ------------------------------------------------------------

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)

# 시간사상별 Fold 번호 저장용 데이터프레임
kf_fold_f = pd.DataFrame({
    'event_id': kf_groups.unique()  # event_id 하나당 한 셀로 생긴 df
})

kf_fold_f['fold_num'] = 0  # 일단 'fold_num 컬럼 0으로 다 채워놓고


# ------------------------------------------------------------
# 3-3. Fold 배정 및 검증
# ------------------------------------------------------------

for fold, (train_idx, valid_idx) in enumerate(  # 반환값은 train과 val의 인덱스 셋
    gkf.split(train_full, groups=kf_groups),  # 분할 대상, 분할때 지킬 그룹기준
    start=1
):
    train_groups = set(kf_groups.iloc[train_idx])  # 집합으로 만들어서
    valid_groups = set(kf_groups.iloc[valid_idx])

    overlap = train_groups & valid_groups  # 겹치는겁 없는지 교집합으로 확인

    # 학습·검증 사상 중복 여부 먼저 확인
    assert len(overlap) == 0, f'Fold {fold}: 시간사상 중복'

    # 검증 Fold에 해당하는 시간사상에 Fold 번호 부여
    kf_fold_f.loc[
        kf_fold_f['event_id'].isin(valid_groups),
        'fold_num'
    ] = fold  # 맞는 인덱스에 가서 번호 붙이기
              # 이후에 순회하면서 번호를 붙여나감

    print(
        f'Fold {fold} | '
        f'train={len(train_idx):,} | '
        f'valid={len(valid_idx):,} | '
        f'train events={len(train_groups):,} | '
        f'valid events={len(valid_groups):,} | '
        f'event overlap={len(overlap)}'
    )


# ------------------------------------------------------------
# 3-4. Fold 배정 결과 최종 검증
# ------------------------------------------------------------

assert kf_fold_f['event_id'].is_unique, 'event_id 중복'
assert kf_fold_f['fold_num'].between(1, N_SPLITS).all(), 'Fold 배정 누락'

assert set(kf_fold_f['event_id']) == set(kf_groups), '시간사상 누락'

# event_id 기준 정렬
kf_fold_f = kf_fold_f.sort_values('event_id').reset_index(drop=True)  # 정렬 후 인덱스 0부터 다시부여

print('\n<Fold별 시간사상 개수>')
print(kf_fold_f['fold_num'].value_counts().sort_index())

display(kf_fold_f)


# ------------------------------------------------------------
# 3-5. Fold 배정 CSV 저장
# ------------------------------------------------------------

kf_fold_f.to_csv(
    PROJECT_ROOT / 'config' / '0_kf_train_full_match.csv',
    index=False
)

print('\nFold 배정 CSV 저장 완료')

KeyError: 'event_id'

In [5]:
# ============================================================
# 1. Tidal / Normal 데이터 분리
# ============================================================

# 조석 영향 의심 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

# Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

print(f'Train: {len(train_full):,}')
print(f'  Tidal: {len(tidal_train):,}')

print(f'\nTest: {len(test):,}')
print(f'  Tidal: {len(tidal_test):,}')

Train: 749,609
  Tidal: 14,251

Test: 114,111
  Tidal: 2,169


In [6]:
# ============================================================
# 2. 독립변수 / 종속변수 분리
# ============================================================

X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_test = tidal_test[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train[target_col]
y_tidal_test = tidal_test[target_col]

# CatBoost 범주형 변수 처리
for X in [
    X_tidal_train, X_tidal_test,
]:
    X['station_id'] = X['station_id'].astype(str)

In [7]:


from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, log_evaluation
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
import gc

# ============================================================
# 3. Tidal 데이터 부스팅모델별 성능 비교
# ============================================================

preds_tidal = {}
results_tidal = {}

for name in ['CatBoost', 'LightGBM', 'XGBoost']:

    print(f'\n========== {name} 학습 ==========')

    # 1. 모델 정의 및 범주형 변수 처리
    if name == 'CatBoost':
        model = CatBoostRegressor(
            random_seed=42,
            cat_features=['station_id'],
            verbose=200
        )

        X_train = X_tidal_train
        X_test = X_tidal_test

    else:
        X_train = X_tidal_train.copy()
        X_test = X_tidal_test.copy()

        # station_id 범주형 자료형 변환
        X_train['station_id'] = X_train['station_id'].astype('category')
        X_test['station_id'] = X_test['station_id'].astype('category')

        if name == 'LightGBM':
            model = LGBMRegressor(
                random_state=42,
                verbosity=1
            )

        elif name == 'XGBoost':
            model = XGBRegressor(
                random_state=42,
                enable_categorical=True,
                eval_metric='rmse'
            )

    # 2. 모델별 학습
    if name == 'LightGBM':
        model.fit(
            X_train, y_tidal_train,
            eval_set=[(X_train, y_tidal_train)],
            eval_metric='rmse',
            callbacks=[log_evaluation(10)]
        )

    elif name == 'XGBoost':
        model.fit(
            X_train, y_tidal_train,
            eval_set=[(X_train, y_tidal_train)],
            verbose=10
        )

    else:
        model.fit(X_train, y_tidal_train)

    # 3. 예측
    pred_test = model.predict(X_test)
    pred_train = model.predict(X_train)

    # 4. RMSE 계산
    rmse_test = root_mean_squared_error(y_tidal_test, pred_test)
    rmse_train = root_mean_squared_error(y_tidal_train, pred_train)

    # 5. 결과 저장
    preds_tidal[name] = pd.Series(
        pred_test, index=X_tidal_test.index
    )

    results_tidal[name] = {
        'Test RMSE': rmse_test,
        'Train RMSE': rmse_train,
        'Gap': rmse_test - rmse_train
    }

    print(f'\n{name} 결과')
    print('Test RMSE:', rmse_test)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_test - rmse_train)

    # 6. 모델 메모리 정리
    del model
    gc.collect()

# 7. 모델별 성능 비교표
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Test RMSE'))



========== CatBoost 학습 ==========
Learning rate set to 0.0623
0:	learn: 1.6771943	total: 230ms	remaining: 3m 49s
200:	learn: 0.3647688	total: 9.32s	remaining: 37s
400:	learn: 0.3083154	total: 19s	remaining: 28.4s
600:	learn: 0.2759927	total: 28.4s	remaining: 18.9s
800:	learn: 0.2534261	total: 40.2s	remaining: 9.99s
999:	learn: 0.2354027	total: 51.6s	remaining: 0us

CatBoost 결과
Test RMSE: 0.35925757219053134
Train RMSE: 0.23576076658529677
Gap: 0.12349680560523457

========== LightGBM 학습 ==========
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002876 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5929
[LightGBM] [Info] Number of data points in the train set: 14251, number of used features: 32
[LightGBM] [Info] Start training from score 1.131862


c:\Users\20162180\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[10]	training's rmse: 0.762341	training's l2: 0.581165
[20]	training's rmse: 0.462833	training's l2: 0.214214
[30]	training's rmse: 0.379341	training's l2: 0.1439
[40]	training's rmse: 0.347612	training's l2: 0.120834
[50]	training's rmse: 0.32812	training's l2: 0.107663
[60]	training's rmse: 0.315271	training's l2: 0.0993959
[70]	training's rmse: 0.304084	training's l2: 0.0924671
[80]	training's rmse: 0.294537	training's l2: 0.0867521
[90]	training's rmse: 0.286393	training's l2: 0.0820208
[100]	training's rmse: 0.278866	training's l2: 0.0777663

LightGBM 결과
Test RMSE: 0.37600386558495336
Train RMSE: 0.27886601288677654
Gap: 0.09713785269817682

========== XGBoost 학습 ==========
[0]	validation_0-rmse:1.29011
[10]	validation_0-rmse:0.34508
[20]	validation_0-rmse:0.29313
[30]	validation_0-rmse:0.26678
[40]	validation_0-rmse:0.24529
[50]	validation_0-rmse:0.22753
[60]	validation_0-rmse:0.21483
[70]	validation_0-rmse:0.20199
[80]	validation_0-rmse:0.18804
[90]	validation_0-rmse:0.17909
[99

,Test RMSE,Train RMSE,Gap
CatBoost,0.359258,0.235761,0.123497
LightGBM,0.376004,0.278866,0.097138
XGBoost,0.386556,0.170794,0.215761


In [8]:

# ============================================================
# 4. 배깅모델을 위한 Tidal 데이터 OneHotEncoding
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

# OHE 전 독립변수 데이터셋
X_tidal_train_num = tidal_train[num_cols_fe]
X_tidal_test_num = tidal_test[num_cols_fe]

# 원핫인코딩
ohe_tidal = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True
)

# Train 기준 학습
ohe_tidal.fit(tidal_train[['station_id']])

# 변환
train_st_id_ohe = ohe_tidal.transform(tidal_train[['station_id']])
test_st_id_ohe = ohe_tidal.transform(tidal_test[['station_id']])

# 수치형 변수 + OHE 결합
X_tidal_train_ohe = hstack([
    X_tidal_train_num,
    train_st_id_ohe
]).tocsr()

X_tidal_test_ohe = hstack([
    X_tidal_test_num,
    test_st_id_ohe
]).tocsr()

print('OHE 전후 데이터 크기')
print(X_tidal_train_num.shape, X_tidal_train_ohe.shape)
print(X_tidal_test_num.shape, X_tidal_test_ohe.shape)

OHE 전후 데이터 크기
(14251, 31) (14251, 40)
(2169, 31) (2169, 40)


In [9]:

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import gc

# ============================================================
# 5. Tidal RF / ExtraTrees 모델별 성능 비교
# ============================================================

# 앞에서 학습한 CatBoost, LightGBM, XGBoost 결과에 추가
# preds_tidal, results_tidal 딕셔너리 유지

for name in ['RandomForest', 'ExtraTrees']:

    print(f'\n========== {name} 학습 ==========')

    # 1. 모델 정의
    if name == 'RandomForest':
        model = RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=2
        )

    elif name == 'ExtraTrees':
        model = ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=2
        )

    # 2. 모델 학습
    model.fit(X_tidal_train_ohe, y_tidal_train)

    # 3. 예측
    pred_test = model.predict(X_tidal_test_ohe)
    pred_train = model.predict(X_tidal_train_ohe)

    # 4. RMSE 계산
    rmse_test = root_mean_squared_error(y_tidal_test, pred_test)
    rmse_train = root_mean_squared_error(y_tidal_train, pred_train)

    # 5. 결과 저장
    preds_tidal[name] = pd.Series(
        pred_test, index=X_tidal_test.index
    )

    results_tidal[name] = {
        'Test RMSE': rmse_test,
        'Train RMSE': rmse_train,
        'Gap': rmse_test - rmse_train
    }

    print(f'\n{name} 결과')
    print('Test RMSE:', rmse_test)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_test - rmse_train)

    # 6. 메모리 정리
    del model
    gc.collect()

# 7. 전체 5개 모델 성능 비교
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Test RMSE'))



========== RandomForest 학습 ==========
building tree 6 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 1 of 100


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 6 concurrent workers.


building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100


[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:   13.8s


building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77

...

building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.2min finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    0.0s



ExtraTrees 결과
Test RMSE: 0.34715941940989614
Train RMSE: 2.8884600960229033e-15
Gap: 0.34715941940989326


[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:    0.0s finished


,Test RMSE,Train RMSE,Gap
ExtraTrees,0.347159,2.888460e-15,0.347159
CatBoost,0.359258,2.357608e-01,0.123497
RandomForest,0.362551,1.280060e-01,0.234545
LightGBM,0.376004,2.788660e-01,0.097138
XGBoost,0.386556,1.707942e-01,0.215761


In [10]:
# ============================================================
# 6. 성능이 우수한 CatBoost + ExtraTrees 앙상블 성능 비교
# ============================================================

# 1. 단순 평균 앙상블 (각각 50%)
pred_ensemble = (
    preds_tidal['CatBoost'] * 0.5
    + preds_tidal['ExtraTrees'] * 0.5
)

# 2. RMSE 계산
rmse_ensemble = root_mean_squared_error(
    y_tidal_test,
    pred_ensemble
)

# 3. 기존 결과에 추가
results_tidal['Ensemble_50_50'] = {
    'Test RMSE': rmse_ensemble,
    'Train RMSE': None,
    'Gap': None
}

# 4. 성능 비교
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Test RMSE'))

,Test RMSE,Train RMSE,Gap
Ensemble_50_50,0.342779,NaN,NaN
ExtraTrees,0.347159,2.888460e-15,0.347159
CatBoost,0.359258,2.357608e-01,0.123497
RandomForest,0.362551,1.280060e-01,0.234545
LightGBM,0.376004,2.788660e-01,0.097138
XGBoost,0.386556,1.707942e-01,0.215761


In [11]:
# ============================================================
# 7. CatBoost + ExtraTrees 앙상블 가중치 최적화
# ============================================================

ensemble_results = []

for w_cat in np.arange(0, 1.001, 0.05):

    pred_ensemble = (
        preds_tidal['CatBoost'] * w_cat
        + preds_tidal['ExtraTrees'] * (1 - w_cat)
    )

    rmse = root_mean_squared_error(
        y_tidal_test,
        pred_ensemble
    )

    ensemble_results.append({
        'CatBoost_weight': round(w_cat, 2),
        'ExtraTrees_weight': round(1 - w_cat, 2),
        'Test RMSE': rmse
    })

ensemble_df = pd.DataFrame(ensemble_results)

display(
    ensemble_df.sort_values('Test RMSE').head(10)
)

,CatBoost_weight,ExtraTrees_weight,Test RMSE
7,0.35,0.65,0.341866
8,0.40,0.60,0.341957
6,0.30,0.70,0.341988
9,0.45,0.55,0.342262
5,0.25,0.75,0.342323
10,0.50,0.50,0.342779
4,0.20,0.80,0.342870
11,0.55,0.45,0.343508
3,0.15,0.85,0.343629
12,0.60,0.40,0.344447


In [ ]:
# 모델 구조 선정: 단일 CatBoost와 Tidal/Normal 분리모델의 K-Fold OOF RMSE를 비교한다. 
# Test 결과도 참고하되, K-Fold에서 더 안정적이고 우수한 구조를 우선한다.


# Early Stopping 적용: K-Fold 또는 기존 단일 홀드아웃 Validation에서 확인한 best_iteration_을 활용해 최종 반복 횟수를 결정한다.
# Tidal과 Normal 각각의 최적 반복 횟수를 별도로 확정